In [53]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pd.set_option('display.max_colwidth', None)

In [19]:
df1 = pd.read_csv('./data/general/news.csv')
df2 = pd.read_csv('./data/general/cleaned_fakenewsdata.csv')
df2['label'] = df2['label'].map({'REAL': 0, 'FAKE': 1})
df2['text'] = df2['text'].str.replace(r'@\s*"', '', regex=True)
df1['label'] = df1['label'].map({'REAL': 0, 'FAKE': 1})
df_comb = pd.concat([df1, df2], ignore_index=True)

In [20]:
from sklearn.utils import resample

# Separate majority and minority classes
df_majority = df_comb[df_comb['label'] == 0]
df_minority = df_comb[df_comb['label'] == 1]

# Downsample majority class
df_majority_downsampled = resample(df_majority,
                                   replace=False,    # sample without replacement
                                   n_samples=len(df_minority),  # to match minority class
                                   random_state=42)  # reproducible results

# Combine minority class with downsampled majority class
df = pd.concat([df_majority_downsampled, df_minority])
df = df.drop(columns=['index', 'title'])
print("Balanced Dataset shape:", df.shape)
df.head()

Balanced Dataset shape: (11774, 2)


,text,label
4799,Like the Titanic’s architects realizing their ...,0
11475,""""""" Democrats on the House Ways and Means Comm...",0
12897,""""""" “ Fresh demands from Russia threatened to ...",0
75,Former GOP presidential hopeful Linsdey Graham...,0
12877,"Indiana, PA (15701) Today Cloudy with showers....",0


In [21]:
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_t, X_val, y_train_t, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [22]:
pipeline = make_pipeline(
    TfidfVectorizer(stop_words='english', max_df=0.7, ngram_range=(1, 2), max_features=5000),
    SGDClassifier(loss='hinge', penalty=None, eta0=1.0, random_state=42)
)

param_grid = {
    'sgdclassifier__alpha': [1e-4, 1e-3, 1e-2],
    'sgdclassifier__learning_rate': ['pa1', 'pa2']
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1'
)

In [23]:
search.fit(X_train_t, y_train_t)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'sgdclassifier__alpha': [0.0001, 0.001, ...], 'sgdclassifier__learning_rate': ['pa1', 'pa2']}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [24]:
y_val_pred = search.predict(X_val)

In [25]:
print("Best Parameters:", search.best_params_)

Best Parameters: {'sgdclassifier__alpha': 0.0001, 'sgdclassifier__learning_rate': 'pa2'}


In [26]:
print("Validation Accuracy:", accuracy_score(y_true=y_val, y_pred=y_val_pred))
print(classification_report(y_true=y_val, y_pred=y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

Validation Accuracy: 0.7186836518046709
              precision    recall  f1-score   support

           0       0.73      0.72      0.72       970
           1       0.71      0.72      0.71       914

    accuracy                           0.72      1884
   macro avg       0.72      0.72      0.72      1884
weighted avg       0.72      0.72      0.72      1884

Confusion Matrix:
 [[696 274]
 [256 658]]


### Trying with new dataset

In [79]:
df_true = pd.read_csv('./data/finance/True.csv')
df_false = pd.read_csv('./data/finance/Fake.csv')
df_true['label'] = 0
df_false['label'] = 1

In [80]:
# describe length of text in both datasets
df_true['text_length'] = df_true['text'].apply(len)
df_false['text_length'] = df_false['text'].apply(len)
print(df_true['text_length'].describe())
print(df_false['text_length'].describe())

count    21417.000000
mean      2383.278517
std       1684.835730
min          1.000000
25%        914.000000
50%       2222.000000
75%       3237.000000
max      29781.000000
Name: text_length, dtype: float64
count    23481.000000
mean      2547.396235
std       2532.884399
min          1.000000
25%       1433.000000
50%       2166.000000
75%       3032.000000
max      51794.000000
Name: text_length, dtype: float64


In [81]:
df_false.nsmallest(5, 'text_length')

,title,text,subject,date,label,text_length
10923,TAKE OUR POLL: Who Do You Think President Trump Should Pick To Replace James Comey?,,politics,"May 10, 2017",1,1
11041,Joe Scarborough BERATES Mika Brzezinski Over “Cheap Shot” At Ivanka Trump: “You don’t have to be so snotty!” [VIDEO],,politics,"Apr 26, 2017",1,1
11190,WATCH TUCKER CARLSON Scorch Sanctuary City Mayor: “Don’t you believe in laws?” [Video],,politics,"Apr 6, 2017",1,1
11225,MAYOR OF SANCTUARY CITY: Trump Trying To Make Us “Fugitive Slave Catchers” [Video],,politics,"Apr 2, 2017",1,1
11236,SHOCKER: Public School Turns Computer Lab Into Mosque…Bars Non-Muslim Students [Video],,politics,"Apr 1, 2017",1,1


In [82]:
df_true.nsmallest(5, 'text_length')

,title,text,subject,date,label,text_length
8970,Graphic: Supreme Court roundup,,politicsNews,"June 16, 2016",0,1
6131,White House: Trump speaks with Egypt's Sisi by phone on Monday,"WASHINGTON (Reuters) - U.S. President Donald Trump was scheduled to speak with Egyptian President Abdel Fattah al-Sisi on Monday, the White House said.",politicsNews,"January 23, 2017",0,152
16059,Spain's cabinet to hold special meeting at 1700 GMT,"MADRID (Reuters) - Spain s cabinet will hold a special meeting at 6 p.m. (1700 GMT), the government said on Tuesday without giving any further details.",worldnews,"October 31, 2017",0,152
19359,UK PM May wants to be a strong friend to the EU,"FLORENCE, Italy (Reuters) - Britain wants to be a strong friend to the European Union and will not block reform, Prime Minister Theresa May said on Friday.",worldnews,"September 22, 2017",0,156
20784,Russia's Putin says we will be able to solve the North Korea crisis by diplomatic means,"VLADIVOSTOK, Russia (Reuters) - Russian President Vladimir Putin said on Thursday that the crisis around North Korea could be resolved by diplomatic means.",worldnews,"September 7, 2017",0,157


In [83]:
df_comb = pd.concat([df_true, df_false], ignore_index=True).drop(columns=['title', 'subject', 'date', 'text_length'])
print("Combined dataset shape:", df_comb.shape)
df_comb.head(1)

Combined dataset shape: (44898, 2)


text  \
0  WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they return from the holidays on Wednesday, lawmakers will begin trying to pass a federal budget in a fight likely to be linked to other issues, such as immigration policy, even as the November congressional election campaigns approach in which Republicans will seek to keep control of Congress. President Donald Trump and his Republicans want a big budget increase in military spending, while Democrats also want proportional increases for non-defense “discretionary” spending on programs that support education, scientific research, infrastructure, public health and environmental protection. “The (Trump) administration has already been willing to say: ‘We’re going to increase non-defense discretionary spending ... by about 7 percent,’” Meadows, chairman of the small but influential House Freedom Caucus, said on the program. “Now, Democrats are saying that’s not enough, we need to give the government a pay raise of 10 to 11 percent. For a fiscal conservative, I don’t see where the rationale is. ... Eventually you run out of other people’s money,” he said. Meadows was among Republicans who voted in late December for their party’s debt-financed tax overhaul, which is expected to balloon the federal budget deficit and add about $1.5 trillion over 10 years to the $20 trillion national debt. “It’s interesting to hear Mark talk about fiscal responsibility,” Democratic U.S. Representative Joseph Crowley said on CBS. Crowley said the Republican tax bill would require the  United States to borrow $1.5 trillion, to be paid off by future generations, to finance tax cuts for corporations and the rich. “This is one of the least ... fiscally responsible bills we’ve ever seen passed in the history of the House of Representatives. I think we’re going to be paying for this for many, many years to come,” Crowley said. Republicans insist the tax package, the biggest U.S. tax overhaul in more than 30 years,  will boost the economy and job growth. House Speaker Paul Ryan, who also supported the tax bill, recently went further than Meadows, making clear in a radio interview that welfare or “entitlement reform,” as the party often calls it, would be a top Republican priority in 2018. In Republican parlance, “entitlement” programs mean food stamps, housing assistance, Medicare and Medicaid health insurance for the elderly, poor and disabled, as well as other programs created by Washington to assist the needy. Democrats seized on Ryan’s early December remarks, saying they showed Republicans would try to pay for their tax overhaul by seeking spending cuts for social programs. But the goals of House Republicans may have to take a back seat to the Senate, where the votes of some Democrats will be needed to approve a budget and prevent a government shutdown. Democrats will use their leverage in the Senate, which Republicans narrowly control, to defend both discretionary non-defense programs and social spending, while tackling the issue of the “Dreamers,” people brought illegally to the country as children. Trump in September put a March 2018 expiration date on the Deferred Action for Childhood Arrivals, or DACA, program, which protects the young immigrants from deportation and provides them with work permits. The president has said in recent Twitter messages he wants funding for his proposed Mexican border wall and other immigration law changes in exchange for agreeing to help the Dreamers. Representative Debbie Dingell told CBS she did not favor linking that issue to 

In [84]:
df = pd.concat([df1, df_comb], ignore_index=True).drop(columns=['index', 'title'])
print("Final Combined Dataset shape:", df.shape)

Final Combined Dataset shape: (51233, 2)


In [85]:
# remove entries where the text length is <200, as they are likely to be noise
df['text_length'] = df['text'].apply(len)
df = df[df['text_length'] >= 200].drop(columns=['text_length'])
print("Dataset shape after removing short texts:", df.shape)

Dataset shape after removing short texts: (49269, 2)


In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_isot_text(text):
    # 1. Remove Publisher tags (ISOT specific)
    text = re.sub(r'^.*? - ', '', text)
    
    # 2. Keep punctuation that matters
    # (Optional: count '!' and '?' here before removing)
    
    # 3. Standard cleaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!\?]', '', text) # Keep text, spaces, !, and ?
    return text

df['text'] = df['text'].apply(clean_isot_text)

In [95]:
print("Sample cleaned text:", df['text'].iloc[30000])

Sample cleaned text: donald trump has been on an insane twitter rampage lately and it s been beyond terrifying for everyone watching trump has not only been crying about fake news and the opposition of his disgusting muslim ban but his narcissism is clearly out of controlthroughout his transition and beginning weeks of his presidency trump has made cnn a target because the news outlet dared to hold his team accountable and the network just hit back at trump again just days after turning down an interview with kellyanne conway because she lacked credibility cnn s jake tapper nailed trump for his recent tweets and made an absolute mockery of the most undeserving potus we ve ever hadtapper comically said that trump s tweets are windows into his soul  right before rattling off several disturbing tweets that trump had posted recently the tweets jumped from criticism of the federal judge who blocked his muslim ban to fake news to negative polls throughout the clip tapper tries to translate t

In [97]:
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_t, X_val, y_train_t, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [96]:
# Use the cleaner, then Vectorize
pipeline = make_pipeline(
    TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=10000),
    SGDClassifier(loss='hinge', penalty=None, eta0=1.0, random_state=42)
)

param_grid = {
    'sgdclassifier__alpha': [1e-4, 1e-3, 1e-2],
    'sgdclassifier__learning_rate': ['pa1', 'pa2']
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1')

In [98]:
search.fit(X_train_t, y_train_t)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'sgdclassifier__alpha': [0.0001, 0.001, ...], 'sgdclassifier__learning_rate': ['pa1', 'pa2']}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [99]:
print("Best Parameters:", search.best_params_)

Best Parameters: {'sgdclassifier__alpha': 0.0001, 'sgdclassifier__learning_rate': 'pa1'}


In [100]:
y_val_pred = search.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_true=y_val, y_pred=y_val_pred))
print(classification_report(y_true=y_val, y_pred=y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

Validation Accuracy: 0.9662565013319802
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      3869
           1       0.97      0.97      0.97      4014

    accuracy                           0.97      7883
   macro avg       0.97      0.97      0.97      7883
weighted avg       0.97      0.97      0.97      7883

Confusion Matrix:
 [[3728  141]
 [ 125 3889]]


In [104]:
# retrain with all training data and test on the test set
pipeline.set_params(**search.best_params_)
pipeline.fit(X_train, y_train)
y_test_pred = pipeline.predict(X_test)
print("Test Accuracy:", accuracy_score(y_true=y_test, y_pred=y_test_pred))
print(classification_report(y_true=y_test, y_pred=y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

Test Accuracy: 0.9696569920844327
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      4842
           1       0.97      0.97      0.97      5012

    accuracy                           0.97      9854
   macro avg       0.97      0.97      0.97      9854
weighted avg       0.97      0.97      0.97      9854

Confusion Matrix:
 [[4678  164]
 [ 135 4877]]


In [106]:
import numpy as np

# Get the feature names from your vectorizer
feature_names = pipeline.named_steps['tfidfvectorizer'].get_feature_names_out()

# Get the coefficients from the SGDClassifier
coefs = pipeline.named_steps['sgdclassifier'].coef_[0]

# Combine and sort
top_real = sorted(zip(coefs, feature_names), reverse=True)[:20]
top_fake = sorted(zip(coefs, feature_names))[:20]

print("Top 20 indicators for REAL:")
for coef, feat in top_real:
    print(f"{feat}: {coef:.4f}")

print("\nTop 20 indicators for FAKE:")
for coef, feat in top_fake:
    print(f"{feat}: {coef:.4f}")

Top 20 indicators for REAL:
image: 13.6631
doesn: 10.6786
didn: 9.1631
october: 8.9856
ap: 8.6010
president trump: 8.4333
wfb: 8.4016
ll: 7.9057
saidthe: 7.8085
isn: 7.7608
don: 7.2962
wasn: 7.2462
nyp: 6.8896
ve: 6.6992
entire: 6.5544
america: 5.8380
wouldn: 5.7517
november: 5.6089
advertisement: 5.5819
couldn: 5.4178

Top 20 indicators for FAKE:
president donald: -11.2991
said: -10.2339
told reuters: -8.7222
trumps: -8.6305
tuesday: -7.5909
est: -7.0362
thursday: -6.5133
friday: -6.2169
said statement: -6.0773
edt: -6.0394
wednesday: -5.8439
doesnt: -5.6246
monday: -5.6130
isnt: -5.2785
partys: -5.2155
thats: -5.2054
obamas: -5.1867
theres: -5.1448
saturday: -5.1229
tuesdays: -5.0674
